<a href="https://colab.research.google.com/github/kuberiitb/shopping_agent/blob/main/notebooks/01_instamart_items_scraping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Instamart Scraper

- This notebook uses instamart categories url and extracts items available on the page
- Information being extracted:
- skuId, displayName, brand, parentProductId, category, imageIds, quantity,  price(currencyCode, mrpPrice,	offerPrice)

In [30]:
# main_url = "https://www.swiggy.com/instamart?entryId=1234&entryName=mainTileEntry4&v=1"

In [31]:
import re
import pickle
import random
import time
import pandas as pd
import requests
import json
from bs4 import BeautifulSoup
from tqdm import notebook

In [32]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Cache-Control": "no-cache",
    "Pragma": "no-cache",
    "Expires": "0",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
    "Connection": "keep-alive"
}

In [33]:
urls = list(set([
    "https://www.swiggy.com/instamart/category-listing?categoryName=Meat%20and%20Seafood&storeId=911033&offset=0&filterName=&taxonomyType=Speciality%20taxonomy%201&showAgeConsent=false",
    "https://www.swiggy.com/instamart/category-listing?categoryName=Atta,%20Rice%20and%20Dal&storeId=911033&offset=0&filterName=&taxonomyType=taxonomy%205&showAgeConsent=false",
    "https://www.swiggy.com/instamart/category-listing?categoryName=Masalas&storeId=911033&offset=0&filterName=&taxonomyType=taxonomy%205&showAgeConsent=false",
    "https://www.swiggy.com/instamart/category-listing?categoryName=Oils%20and%20Ghee&storeId=911033&offset=0&filterName=&taxonomyType=taxonomy%205&showAgeConsent=false",
    "https://www.swiggy.com/instamart/category-listing?categoryName=Cereals%20and%20Breakfast&storeId=911033&offset=0&filterName=&taxonomyType=taxonomy%205&showAgeConsent=false",
    "https://www.swiggy.com/instamart/category-listing?categoryName=Cereals%20and%20Breakfast&storeId=911033&offset=0&filterName=&taxonomyType=taxonomy%205&showAgeConsent=false",
    "https://www.swiggy.com/instamart/category-listing?categoryName=Ice%20Creams%20and%20Frozen%20Desserts&storeId=911033&offset=0&filterName=&taxonomyType=taxonomy%2010&showAgeConsent=false",
    "https://www.swiggy.com/instamart/category-listing?categoryName=Chocolates&storeId=911033&offset=0&filterName=&taxonomyType=taxonomy%2010&showAgeConsent=false",
    "https://www.swiggy.com/instamart/category-listing?categoryName=Biscuits%20and%20Cakes&storeId=911033&offset=0&filterName=&taxonomyType=taxonomy%2010&showAgeConsent=false",
    "https://www.swiggy.com/instamart/category-listing?categoryName=Tea,%20Coffee%20and%20Milk%20drinks&storeId=911033&offset=0&filterName=&taxonomyType=taxonomy%2010&showAgeConsent=false",
    "https://www.swiggy.com/instamart/category-listing?categoryName=Sauces%20and%20Spreads&storeId=911033&offset=0&filterName=&taxonomyType=taxonomy%2010&showAgeConsent=false",
    "https://www.swiggy.com/instamart/category-listing?categoryName=Noodles,%20Pasta,%20Vermicelli&storeId=911033&offset=0&filterName=&taxonomyType=taxonomy%2010&showAgeConsent=false",
    "https://www.swiggy.com/instamart/category-listing?categoryName=Cleaners%20and%20Repellents&storeId=911033&offset=0&filterName=&taxonomyType=IM%20Meatsy&showAgeConsent=false",
    "https://www.swiggy.com/instamart/category-listing?categoryName=Bath%20and%20Body&storeId=911033&offset=0&filterName=&taxonomyType=taxonomy%2014&showAgeConsent=false",
    "https://www.swiggy.com/instamart/category-listing?categoryName=Makeup&storeId=911033&offset=0&filterName=&taxonomyType=taxonomy%2014&showAgeConsent=false",
    "https://www.swiggy.com/instamart/category-listing?categoryName=Feminine%20Hygiene&storeId=911033&offset=0&filterName=&taxonomyType=taxonomy%2014&showAgeConsent=false",
    "https://www.swiggy.com/instamart/category-listing?categoryName=Baby%20Care&storeId=911033&offset=0&filterName=&taxonomyType=taxonomy%2014&showAgeConsent=false",
    "https://www.swiggy.com/instamart/category-listing?categoryName=Health%20and%20Pharma&storeId=911033&offset=0&filterName=&taxonomyType=taxonomy%2014&showAgeConsent=false",
    "https://www.swiggy.com/instamart/category-listing?categoryName=Skincare&storeId=911033&offset=0&filterName=&taxonomyType=taxonomy%2014&showAgeConsent=false",
    "https://www.swiggy.com/instamart/category-listing?categoryName=Fashion&storeId=911033&offset=0&filterName=&taxonomyType=IM%20Meatsy&showAgeConsent=false",
    "https://www.swiggy.com/instamart/category-listing?categoryName=Electronics%20and%20Appliances&storeId=911033&offset=0&filterName=&taxonomyType=IM%20Meatsy&showAgeConsent=false",
    "https://www.swiggy.com/instamart/category-listing?categoryName=Fresh%20Fruits&storeId=911033&offset=0&filterName=&taxonomyType=Speciality%20taxonomy%201&showAgeConsent=false",
    "https://www.swiggy.com/instamart/category-listing?categoryName=Fresh%20Vegetables&storeId=911033&offset=0&filterName=&taxonomyType=Speciality%20taxonomy%201&showAgeConsent=false",
    "https://www.swiggy.com/instamart/collection-listing?collectionId=72456&custom_back=true",
]))


In [34]:
def get_data(url):
  response = requests.get(url, headers=headers)
  soup = BeautifulSoup(response.text, "html.parser")
  scripts = soup.find_all("script")

  raw_json = None

  for script in scripts:
      if script.string and "ItemCollectionCard" in script.string:
          raw_json = extract_first_json(
              script.string,
              '{"@type":"type.googleapis.com/swiggy.im.v1.ItemCollectionCard"'
          )
          break

  data = json.loads(raw_json)['items']
  return data


In [35]:
def extract_first_json(text, start_key):
    start = text.find(start_key)
    if start == -1:
        return None

    brace_count = 0
    in_string = False
    escape = False

    for i in range(start, len(text)):
        char = text[i]

        if char == '"' and not escape:
            in_string = not in_string

        if not in_string:
            if char == '{':
                brace_count += 1
            elif char == '}':
                brace_count -= 1

        if char == '\\' and not escape:
            escape = True
        else:
            escape = False

        if brace_count == 0:
            return text[start:i+1]

    return None

def extract_quantity(title):
    match = re.search(r'(\d+\s?(g|gm|kg|ml|l))', title.lower())
    return match.group(0) if match else None

def extract_items_from_dict(d, keys):
  out = {}
  for key in keys:
    if type(key)==str:
      out[key] = d[key]
    elif type(key)==dict:
      # print("dict key", key)

      for mainkey, subkeys in key.items():
        out[mainkey] = {}
        # print("mainkey", mainkey)
        # print("subkeys", subkeys)
        temp = {}
        for mainkeydata in d[mainkey]:
          for subkey in subkeys:
            temp[subkey] = mainkeydata[subkey]
          out[mainkey] = temp
  return out



In [36]:
#base image url https://instamart-media-assets.swiggy.com/swiggy/image/upload/fl_lossy,f_auto,q_auto,h_600/

In [37]:
try:
  #load existing data(if available)
  main_data = pickle.load(open('main_data.pkl','rb'))
  print(len(main_data))
except:
  main_data = {}

for idxx in notebook.tqdm(range(1, 101)):
  if idxx%10==0:
    print(f"{idxx} done. datasize {len(main_data)}. Waiting for 2 sec.")
    time.sleep(0.5)
  else:
    time.sleep(0.1)

  # pick a category url randomly
  url = random.sample(urls,1)[0]
  try:
    data = get_data(url)
  except:
    continue

  extracted_data = []
  for item in data:
    out = extract_items_from_dict(item, ['displayName','brand', 'parentProductId', {'variations':['category', 'price', 'imageIds', 'skuId','quantityDescription']}])
    extracted_data.append(out)

  df = pd.DataFrame(extracted_data)

  #convert hierarchical columns to flat
  new_cols = pd.json_normalize(df['variations'])

  df = df.drop('variations',axis=1).join(new_cols)
  df['imageIds'] = df['imageIds'].apply(lambda x:x[0] if type(x)==list else x)
  df = df.loc[:, [x for x in df.columns if ('price' not in x) or (x in ['price.mrp.currencyCode','price.mrp.units','price.offerPrice.units'])] ]
  df = df.rename(columns={'price.mrp.currencyCode':'currencyCode',
                    'price.mrp.units':'mrp',
                    'price.offerPrice.units':'offerPrice'
  })

  #updating records if there is new information(some data has mrp information missing)
  for _, k in df.iterrows():
    data_dict = k.to_dict()
    if (data_dict['skuId'] not in main_data) or ((data_dict['skuId'] in main_data) and ('mrp' not in main_data.get(data_dict['skuId'])) ):
      main_data[data_dict['skuId']] = data_dict

print("Items count", len(main_data))
print("Items with images", len([x for _, x in main_data.items() if x.get('imageIds')]))
print("Items with MRP", len([x for _, x in main_data.items() if x.get('mrp')]))

pickle.dump(main_data, open('main_data.pkl','wb'))
print("Dump updated")

317


  0%|          | 0/100 [00:00<?, ?it/s]

10 done. datasize 318. Waiting for 2 sec.
20 done. datasize 318. Waiting for 2 sec.
30 done. datasize 318. Waiting for 2 sec.
40 done. datasize 318. Waiting for 2 sec.
50 done. datasize 318. Waiting for 2 sec.
60 done. datasize 318. Waiting for 2 sec.
70 done. datasize 318. Waiting for 2 sec.
80 done. datasize 318. Waiting for 2 sec.
90 done. datasize 319. Waiting for 2 sec.
100 done. datasize 319. Waiting for 2 sec.
Items count 319
Items with images 236
Items with MRP 178
Dump updated


## Check data

In [49]:
df = pd.DataFrame(main_data).T
df = df.loc[~df['mrp'].isna(),:]
df.head()

,displayName,brand,parentProductId,category,skuId,quantityDescription,imageIds,currencyCode,mrp,offerPrice
3K4IJLW5DH,Onion (Eerulli),Fruits and Vegetables,TXX0TEK4FU,Vegetables,3K4IJLW5DH,1 kg,NI_CATALOG/IMAGES/CIW/2026/3/24/a571cc74-b070-...,INR,36,29
21B8Y0FOTH,Wellbeing Nutrition 100% Whey Protein Isolate ...,Wellbeing Nutrition,MKEKI5DU81,Protein and Sports Nutrition,21B8Y0FOTH,1 kg,NI_CATALOG/IMAGES/ciw/2025/12/18/7979dba5-393a...,INR,4499,4049
XUIZD4U4I9,Wellbeing Nutrition 100% Whey Protein Isolate ...,Wellbeing Nutrition,46X8P6DPKV,Protein and Sports Nutrition,XUIZD4U4I9,1 kg,NI_CATALOG/IMAGES/ciw/2025/12/18/d9a54e4c-2f71...,INR,4499,4049
JDQGR7N0NV,"HEEVA All Natural Crunchy Peanut Butter, Unswe...",HEEVA,1YR21YNBEB,Peanut Butters,JDQGR7N0NV,1 kg,NI_CATALOG/IMAGES/ciw/2026/3/11/c9f11f52-3e95-...,INR,449,337
PBKBUBVCVD,Ariel Lavender Power Gel Liquid Detergent Top ...,Ariel,WBM1MDB8WT,Detergents,PBKBUBVCVD,3.6 kg,NI_CATALOG/IMAGES/CIW/2026/1/23/30c134b0-c1a7-...,INR,765,597


In [81]:
main_data['3K4IJLW5DH']

{'displayName': 'Onion (Eerulli)',
 'brand': 'Fruits and Vegetables',
 'parentProductId': 'TXX0TEK4FU',
 'category': 'Vegetables',
 'imageIds': 'NI_CATALOG/IMAGES/CIW/2026/3/24/a571cc74-b070-43d5-8578-d414312f0551_2116_1.jpg',
 'skuId': '3K4IJLW5DH',
 'quantityDescription': '1 kg',
 'currencyCode': 'INR',
 'mrp': '36',
 'offerPrice': '29'}

## Appendix: Can we get skuId and extract details from respective page?

In [41]:
# https://www.swiggy.com/instamart/item/MQBV46M8S1

In [42]:
# skuId

In [43]:
url = "https://www.swiggy.com/instamart/category-listing?categoryName=Tea,%20Coffee%20and%20Milk%20drinks&storeId=911033&offset=0&filterName=&taxonomyType=taxonomy%2010&showAgeConsent=false"

response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.text, "html.parser")
sku_ids = []

scripts = soup.find_all("script")

for script in scripts:
    content = script.string or script.get_text()
    if not content:
        continue

    matches = re.findall(r'"productId"\s*:\s*"([^"]+)"', content)
    for match in matches:
        sku_ids.append(match)

print(sku_ids)

['WCMH0YFUYQ', 'G7126YK69Y', 'CNTE88XFKU', '22FIVAHNBW', 'Y25FNX1JSF', 'BZ1W3CXZK1', 'CGF02UH6CX', 'ELHBGXIHW2', 'JXC2QAIW5I', 'YJ41CAJVRZ', '9MU6G6ZJ8J', 'H13W08ZDJ8', 'MQBV46M8S1', 'UOYZHA0BSK', '6QTXG7C22N', 'VOVUJ5KK9O', '2IZ1J9FBZW', 'QRUTIPWVB6', '69VX0TBI7W', 'WFI8BWSLZC', 'I04W6NQ0MR', 'FWVTVHKS75', 'OHHVBU1MJS', '8PR40TX7TS', '3Q60ITM7B0', 'M2Z6XF9M54', 'WCMH0YFUYQ', 'G7126YK69Y', 'CNTE88XFKU', '22FIVAHNBW', 'Y25FNX1JSF', 'BZ1W3CXZK1', 'CGF02UH6CX', 'ELHBGXIHW2', 'JXC2QAIW5I', 'YJ41CAJVRZ', '9MU6G6ZJ8J', 'H13W08ZDJ8', 'MQBV46M8S1', 'UOYZHA0BSK', '6QTXG7C22N', 'VOVUJ5KK9O', '2IZ1J9FBZW', 'QRUTIPWVB6', '69VX0TBI7W', 'WFI8BWSLZC', 'I04W6NQ0MR', 'FWVTVHKS75', 'OHHVBU1MJS', '8PR40TX7TS', '3Q60ITM7B0', 'M2Z6XF9M54']


In [44]:
item_base_url = "https://www.swiggy.com/instamart/item/"
item_url = item_base_url+"5KBTO9JF27"

item_url = "https://www.swiggy.com/instamart/item/5KBTO9JF27"

In [45]:
response.text

'<!DOCTYPE html><html lang="en" class="web fonts-loaded"><head><script type="text/javascript">window.chunkUrl="https://instamart-media-assets.swiggy.com/swiggy/raw/upload/dash-front-assets/js/"</script>  <link rel="preconnect" href="//instamart-media-assets.swiggy.com">  <link rel="preconnect" href="//analytics.swiggy.com">  <link rel="preconnect" href="//payments.swiggy.com">  <link rel="preconnect" href="//media-assets.swiggy.com">   <link rel="dns-prefetch" href="//instamart-media-assets.swiggy.com">  <link rel="dns-prefetch" href="//analytics.swiggy.com">  <link rel="dns-prefetch" href="//media-assets.swiggy.com">  <script>!function(e,t,a,n,m){e[n]=e[n]||[],e[n].push({"gtm.start":(new Date).getTime(),event:"gtm.js"});var g=t.getElementsByTagName(a)[0],r=t.createElement(a);r.async=!0,r.src="https://www.googletagmanager.com/gtm.js?id=GTM-KCZG4Z2",g.parentNode.insertBefore(r,g)}(window,document,"script","dataLayer")</script> <meta name="viewport" content="height=device-height,width=de

In [46]:
response = requests.get(item_url, headers=headers)
soup = BeautifulSoup(response.text, "html.parser")

scripts = soup.find_all("script")

# "sc-gEvEer gmdlfz _1iFYi"

import re

products = []

for script in soup.find_all("script"):
    content = script.string or script.get_text()
    if not content:
        continue
    print(content)

window.chunkUrl="https://instamart-media-assets.swiggy.com/swiggy/raw/upload/dash-front-assets/js/"
!function(e,t,a,n,m){e[n]=e[n]||[],e[n].push({"gtm.start":(new Date).getTime(),event:"gtm.js"});var g=t.getElementsByTagName(a)[0],r=t.createElement(a);r.async=!0,r.src="https://www.googletagmanager.com/gtm.js?id=GTM-KCZG4Z2",g.parentNode.insertBefore(r,g)}(window,document,"script","dataLayer")
!function(c,f){var t,o,i,e=[],r={passive:!0,capture:!0},n=new Date,a="pointerup",u="pointercancel";function p(n,e){t||(t=e,o=n,i=new Date,w(f),s())}function s(){0<=o&&o<i-n&&(e.forEach(function(n){n(o,t)}),e=[])}function l(n){if(n.cancelable){var e=(1e12<n.timeStamp?new Date:performance.now())-n.timeStamp;"pointerdown"==n.type?function(n,e){function t(){p(n,e),i()}function o(){i()}function i(){f(a,t,r),f(u,o,r)}c(a,t,r),c(u,o,r)}(e,n):p(e,n)}}function w(e){["click","mousedown","keydown","touchstart","pointerdown"].forEach(function(n){e(n,l,r)})}w(c),self.perfMetrics=self.perfMetrics||{},self.perfM

In [48]:
d1_out = []
for item in data:
  out = extract_items_from_dict(item, ['displayName','brand', 'parentProductId', {'variations':['category', 'imageIds', 'skuId','quantityDescription']}])
  d1_out.append(out)